# Word Embedding
This notebook introduces word embeddings, mathematics, and implementation.

***Word Embedding*** is a type of word representation that allows words to be represented as vectors in a continuous vector space, 

capturing `semantic similarity`. **Word2Vec** and **GloVe** are popular word embedding models. Word embeddings transform words into 

numerical vectors in a continuous vector space. The key idea is that words with similar meanings will have similar vector 

representations (i.e., they will be close to each other in the vector space).



### Vector Space Model

In a vector space model, each word is mapped to a point (a vector) in a high-dimensional space. The dimensions of this space are 

not directly interpretable as features like "color" or "size" but rather capture abstract semantic and syntactic properties learned from the text.

A word $w$ is represented by a vector $\vec{v}_w \in \mathbb{R}^d$, where $d$ is the dimensionality of the embedding space (typically between 50 and 300).


### Similarity Measures

To quantify how "similar" two words are based on their vector representations, we use similarity measures. The most common one is **Cosine Similarity**.

Cosine Similarity measures the cosine of the angle between two vectors. A cosine similarity of 1 means the vectors are identical in direction (highly similar), 

0 means they are orthogonal (no similarity), and -1 means they are opposite in direction (highly dissimilar).

For two word vectors $\vec{A}$ and $\vec{B}$:

$\text{Cosine Similarity}(\vec{A}, \vec{B}) = \frac{\vec{A} \cdot \vec{B}}{|\vec{A}| |\vec{B}|} = \frac{\sum_{i=1}^d A_i B_i}{\sqrt{\sum_{i=1}^d A_i^2} \sqrt{\sum_{i=1}^d B_i^2}}$

Where $\vec{A} \cdot \vec{B}$ is the dot product of vectors $\vec{A}$ and $\vec{B}$ and $|\vec{A}|$ and $|\vec{B}|$ are the Euclidean norms (magnitudes) of vectors $\vec{A}$ and $\vec{B}$, respectively.


### Neural Network Basics for Word2Vec
Understanding the neural network basics behind Word2Vec involves diving into how it learns vector representations of words using a simple

neural architecture. Word2Vec has two main architectures: **Continuous Bag of Words (CBOW)** and **Skip-gram**

**1. Skip-gram Architecture**: Given a **center word**, predict the surrounding context words.

For example, in the sentence "The cat sat on the mat" for the center word **"cat"**, the model should predict context words like "The", "sat".

**Training Data:** From a sentence, we extract `(center, context)` pairs. For example, with a size of 2, `input: cat` and `Targets: The, sat`

***Neural Network Structure for Skip-gram***

The original Word2Vec neural network is a shallow neural network with one hidden layer $\text{Input: One-hot vector}$ of center word(V-dimensional),

$\text{Hidden Layer: Weights matrix W}$ $(V \times N)$ and $\text{Output: Probability distribution}(N \times V)$ over vocabulary (predicting context words)

where $V = \text{Vocabulary size}, N = \text{Embedding size}$ (e.g., 100, 300)

***Steps:***

**1. Input:** One-hot encode the center word.

**2. Hidden layer:** Multiply one-hot vector by weights → gives a dense vector (word embedding). This is basically selecting the corresponding 

row from matrix $W$.

**3. Output layer:** Multiply embedding by W' to get a score for every word in vocabulary.

**4. Softmax:** Turn the scores into probabilities.

**5. Loss function:** Use cross-entropy loss (or negative sampling) to compare predicted vs actual context word.

### Python Code: Skip-gram with Softmax (Manual Example)
Let’s say the vocab is: **[the, cat, sat, on, mat]** and we want $\text{embedding size =} 2$, and word is 'cat' that One-hot vector is $[0, 1, 0, 0, 0]$

In [ ]:
import torch
import torch.nn.functional as F

# Vocabulary and one-hot encoder
vocab = ['the', 'cat', 'sat', 'on', 'mat']
# a dict initalized with index of words in the vocab.
word_to_ix = {word: i for i, word in enumerate(vocab)}

#1. One-hot vector for 'cat'
center_word = 'cat'
x = torch.zeros(len(vocab))
x[word_to_ix[center_word]] = 1

print("One-hot encoded vector for 'cat', this is input vector:\nx=", x.tolist())
print()

# Initialize weights
V = len(vocab)  # vocab size = 5
N = 2           # embedding size = 2

torch.manual_seed(42)  # for reproducibility

W = torch.randn(V, N, requires_grad=True)        # random input weight matrix (5 x 2)
W_prime = torch.randn(N, V, requires_grad=True)  # random output weight matrix (2 x 5)

# Forward pass
h = x @ W         # hidden layer: (1 x 5) @ (5 x 2) → (1 x 2)
u = h @ W_prime   # output layer: (1 x 2) @ (2 x 5) → (1 x 5)
print("Output scores (before softmax):\nu=", u.detach().numpy())
# Softmax
y_pred = F.softmax(u, dim=0)

# Print results
print("\nWord Embedding for 'cat':", h.detach().numpy())
print()
print("Predicted probabilities:", y_pred.detach().numpy())

# Assume target context word is 'sat'
target_word = 'sat'
target_index = word_to_ix[target_word]

# Compute loss: cross entropy (manually)
loss = -torch.log(y_pred[target_index])
print("\nLoss (cross-entropy):", loss.item())


One-hot encoded vector for 'cat', this is input vector:
x= [0.0, 1.0, 0.0, 0.0, 0.0]

Output scores (before softmax):
u= [ 0.34606755  0.4942952   0.4485471  -0.5725922  -0.404767  ]

Word Embedding for 'cat': [0.23446237 0.23033303]

Predicted probabilities: [0.24162073 0.28022614 0.26769516 0.0964196  0.11403835]

Loss (cross-entropy): 1.317906379699707


### Output anaisis
***Word Embedding for 'cat'*** is $[0.23446237, 0.23033303]$. This is the learned 2D embedding of the word 'cat'. It's the row at index 1 

in the matrix W (shape [5, 2]). The model is learning to map 'cat' to this point in 2D space. With more training, this vector will 

move to encode similarity to its neighbors (e.g., closer to 'sat' if that’s its frequent context).

***Output scores before softmax***, vector $u$ is $[ 0.34606755,  0.4942952,  0.4485471, -0.5725922, -0.404767 ]$. These are the raw logits 

computed by $u = h.W^\prime$ — how strongly the center word 'cat' is associated with each output word.

'cat' (input) gives highest score to 'cat' (index 1) and 'sat' (index 2). This suggests the model has begun learning that 'sat' is in its context!

***Softmax probabilities*** is $[0.24162073, 0.28022614, 0.26769516, 0.0964196, 0.11403835]$. Now these scores are normalized into 

probabilities (they sum to 1). Probabilities indicate how likely each word is to be in the context of 'cat'.
<div>

<table border="1" style="border-collapse: collapse; margin-left:50px">
<tr>
<td width='200px'>Word</td>
<td width='300px'>Probability</td>
<tr><td>the</td><td>24.16%</td>
<tr><td>cat</td><td>28.02%</td>
<tr><td>sat</td><td>26.77%</td>
<tr><td>on</td><td>9.64%</td>
<tr><td>mat</td><td>11.40%</td>
</table>
</div>

This is good sign, 'sat' is getting a fairly high probability (26.77%) — if this is the actual context word, the model is doing reasonably well 

after 0 or 1 training steps.



***Loss:***
Cross-entropy $loss = 1.3179$ Cross-entropy is $L=−log⁡(P(\text{true context word}))$. In this case, it's $−log(0.2677)=1.3179$ 

(i.e., the predicted probability for 'sat'). A perfect prediction would have $loss ≈ 0$.

This moderate value $≈ 1.3$ shows the model is partially confident, but still has room to improve.


***Optimization tricks:***

Use gradient descent to update WW and W′W′ to minimize the loss.

For large vocabularies, use negative sampling to speed up the softmax.

### Full Example with Manual Gradient Descent (1 training step)

In [17]:
import torch
import torch.nn.functional as F

# Vocabulary setup
vocab = ['the', 'cat', 'sat', 'on', 'mat']
word_to_ix = {word: i for i, word in enumerate(vocab)}

# Parameters
V = len(vocab)  # vocab size = 5
N = 2           # embedding size = 2
lr = 0.1        # learning rate

# Seed for reproducibility
torch.manual_seed(42)

# Initialize weights
W = torch.randn(V, N, requires_grad=True)       # Input embedding matrix
W_prime = torch.randn(N, V, requires_grad=True) # Output weight matrix

# One training example: ('cat' -> 'sat')
center_word = 'cat'
target_word = 'sat'
center_ix = word_to_ix[center_word]
target_ix = word_to_ix[target_word]

# One-hot vector for center word
x = torch.zeros(V)
x[center_ix] = 1

# --- Forward pass ---
h = x @ W                  # hidden layer output (1 x 2)
u = h @ W_prime            # raw output scores (1 x 5)
y_pred = F.softmax(u, dim=0)

# --- Compute loss ---
loss = -torch.log(y_pred[target_ix])
print("Loss before update:", loss.item())

# --- Backward pass ---
loss.backward()

# --- Update weights manually (gradient descent step) ---
with torch.no_grad():
    W -= lr * W.grad
    W_prime -= lr * W_prime.grad

    # Clear gradients
    W.grad.zero_()
    W_prime.grad.zero_()

# --- Forward pass again to show improvement ---
h_new = x @ W
u_new = h_new @ W_prime
y_pred_new = F.softmax(u_new, dim=0)
loss_new = -torch.log(y_pred_new[target_ix])

print(f"Loss after 1 update:{loss_new.item()}")
print("\nUpdated word embedding for 'cat':", h_new.detach().numpy())


Loss before update: 1.317906379699707
Loss after 1 update:1.2558692693710327

Updated word embedding for 'cat': [0.30773565 0.24592413]


The exact numbers will vary slightly due to weight initialization. But you should see that the loss decreases after the gradient step.

# If this topic was helpful to you, please give me a star ⭐.